#Name: Rishil Abhijit Jalisatgi
#SRN: PES2UG23CS482
#SEC: H

In [1]:
# Cell 1: Setup & Installations
%pip install python-dotenv rank-bm25 sentence-transformers langchain langchain-google-genai langchain-community langchain-huggingface numpy pandas

from dotenv import load_dotenv
import os

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

print("Setup complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
Enter your Google API Key: ··········
Setup complete.


In [2]:
# Cell 2: Imports
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from IPython.display import display, Markdown

In [3]:
# Cell 3: Document Corpus Setup
# 10 documents: Includes distinct sub-topics (attention/training) and technical jargon (ELMo, HNSW)
corpus = [
    "Attention mechanisms allow neural networks to dynamically weigh the importance of different input tokens when processing sequences.",
    "Self-attention computes a weighted average of token representations based on query, key, and value vectors.",
    "Multi-head attention runs several self-attention operations in parallel to capture different representational subspaces.",
    "AdamW is an optimization technique that decouples weight decay from the gradient update to improve model generalization.",
    "Stochastic Gradient Descent (SGD) updates model parameters using the gradient of the loss with respect to a single or a small batch of examples.",
    "Learning rate schedulers, like Cosine Annealing, adjust the step size during training to help the optimizer escape local minima.",
    "The ELMo architecture uses bidirectional LSTMs to generate context-sensitive word embeddings, unlike static embeddings like Word2Vec.",
    "Retrieval-Augmented Generation (RAG) grounds language models in external knowledge bases to reduce hallucinations and improve factual accuracy.",
    "Vector databases index dense embeddings using algorithms like HNSW for fast approximate nearest neighbor search.",
    "Cross-encoders process the query and document simultaneously through a Transformer block, yielding highly accurate relevance scores but at a high computational cost."
]

print(f"Corpus loaded with {len(corpus)} documents.")

Corpus loaded with 10 documents.


In [4]:
# Cell 4: Hybrid Retrieval Implementation
class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k
        self.doc_ids = list(range(len(corpus)))

        # Initialize BM25
        tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized_corpus)

        # Initialize SBERT (Dense Retriever)
        self.sbert = SentenceTransformer('all-MiniLM-L6-v2')
        self.corpus_embeddings = self.sbert.encode(corpus, convert_to_tensor=True)
        print("HybridRetriever initialized successfully.")

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        # 1. BM25 Scoring & Ranking
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranked_indices = np.argsort(bm25_scores)[::-1]
        bm25_ranks = {idx: rank + 1 for rank, idx in enumerate(bm25_ranked_indices)}

        # 2. SBERT Scoring & Ranking
        query_embedding = self.sbert.encode(query, convert_to_tensor=True)
        cosine_scores = util.cos_sim(query_embedding, self.corpus_embeddings)[0].cpu().numpy()
        sbert_ranked_indices = np.argsort(cosine_scores)[::-1]
        sbert_ranks = {idx: rank + 1 for rank, idx in enumerate(sbert_ranked_indices)}

        # 3. Reciprocal Rank Fusion (RRF)
        results = []
        for idx in self.doc_ids:
            r_bm25 = bm25_ranks[idx]
            r_sbert = sbert_ranks[idx]

            # RRF Formula
            rrf_score = (1.0 / (self.k + r_bm25)) + (1.0 / (self.k + r_sbert))

            results.append({
                "doc_id": idx,
                "rrf_score": rrf_score,
                "bm25_rank": r_bm25,
                "sbert_rank": r_sbert,
                "text": self.corpus[idx],
                "dense_score": float(cosine_scores[idx]) # Kept for Naive RAG comparison later
            })

        # Sort by RRF score descending
        results = sorted(results, key=lambda x: x["rrf_score"], reverse=True)
        return results[:top_k]

    def naive_dense_retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        """Helper method specifically for the Naive RAG baseline."""
        query_embedding = self.sbert.encode(query, convert_to_tensor=True)
        cosine_scores = util.cos_sim(query_embedding, self.corpus_embeddings)[0].cpu().numpy()
        ranked_indices = np.argsort(cosine_scores)[::-1]

        return [{"doc_id": idx, "text": self.corpus[idx], "dense_score": float(cosine_scores[idx])}
                for idx in ranked_indices[:top_k]]

# Instantiate the retriever
retriever = HybridRetriever(corpus)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HybridRetriever initialized successfully.


In [5]:
# Cell 5: Cross-Encoder Re-Ranker
cross_encoder_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    # Form pairs of (Original Query, Document Text)
    pairs = [[query, doc["text"]] for doc in candidates]

    # Predict cross-encoder scores
    scores = cross_encoder_model.predict(pairs)

    # Attach scores to candidates and sort
    for idx, doc in enumerate(candidates):
        doc["cross_encoder_score"] = float(scores[idx])

    reranked = sorted(candidates, key=lambda x: x["cross_encoder_score"], reverse=True)
    return reranked[:top_k]

print("Cross-encoder ready.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-encoder ready.


In [10]:
# Cell 6: Query Expansion (HyDE)
# Using temperature=0.0 for deterministic, factual hypothetical documents
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

hyde_prompt = PromptTemplate.from_template(
    "You are an AI/ML expert. Write a brief, factual, 1-2 sentence hypothetical paragraph that directly answers the following question. Do not include introductory filler.\n\nQuestion: {query}\n\nHypothetical Answer:"
)

hyde_chain = hyde_prompt | llm

def generate_hyde_query(query: str) -> str:
    response = hyde_chain.invoke({"query": query})
    hypothetical_doc = response.content.strip()
    return hypothetical_doc

print("HyDE setup complete.")

HyDE setup complete.


In [11]:
# Cell 7: Final Generation Setup
qa_prompt = PromptTemplate.from_template(
    "Answer the user's question based strictly on the provided context. If the context does not contain the answer, say 'I don't know based on the context.'\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
)
qa_chain = qa_prompt | llm

In [12]:
# Cell 8: End-to-End Pipelines (Naïve vs Advanced)

def naive_rag(user_query: str) -> dict:
    """Baseline: Dense-only retrieval (SBERT), no expansion, no re-ranking."""
    # 1. Retrieve
    top_docs = retriever.naive_dense_retrieve(user_query, top_k=3)
    context = "\n".join([doc["text"] for doc in top_docs])

    # 2. Generate
    response = qa_chain.invoke({"context": context, "query": user_query})

    return {
        "answer": response.content.strip(),
        "top_doc": top_docs[0]["text"] if top_docs else "None"
    }

def advanced_rag(user_query: str) -> dict:
    """Full pipeline: Query Expansion (HyDE) -> Hybrid Retrieval -> Re-Ranking -> Generation"""
    # 1. Query Expansion (HyDE)
    expanded_query = generate_hyde_query(user_query)

    # 2. Hybrid Retrieval (Using the expanded query to search!)
    # Retrieving top 5 to give the cross-encoder a good pool to rank
    candidates = retriever.retrieve(expanded_query, top_k=5)

    # 3. Cross-Encoder Re-Ranking (Using the ORIGINAL query!)
    reranked_docs = rerank(user_query, candidates, top_k=3)

    # 4. Generate
    context = "\n".join([doc["text"] for doc in reranked_docs])
    response = qa_chain.invoke({"context": context, "query": user_query})

    return {
        "answer": response.content.strip(),
        "top_doc": reranked_docs[0]["text"] if reranked_docs else "None"
    }

print("Pipelines defined.")

Pipelines defined.


In [13]:
# Cell 9: Run Experiment and Generate Markdown Table
test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "why are vector databases fast for nearest neighbor searches?" # Custom query mapped to HNSW doc
]

results = []

for q in test_queries:
    print(f"Processing: '{q}'...")
    naive_result = naive_rag(q)
    adv_result = advanced_rag(q)

    is_different = "Yes" if naive_result["top_doc"] != adv_result["top_doc"] else "No"

    results.append({
        "Query": f"`\"{q}\"`",
        "Naïve RAG Top Doc": naive_result["top_doc"],
        "Advanced RAG Top Doc": adv_result["top_doc"],
        "Are they different?": is_different
    })

# Format as Markdown Table
df = pd.DataFrame(results)
markdown_table = df.to_markdown(index=False)

display(Markdown("### Part 6 — Comparison Experiment Results"))
display(Markdown(markdown_table))

Processing: 'how do transformers encode meaning?'...
Processing: 'optimization techniques for training'...
Processing: 'why are vector databases fast for nearest neighbor searches?'...


### Part 6 — Comparison Experiment Results

| Query                                                            | Naïve RAG Top Doc                                                                                                                                                     | Advanced RAG Top Doc                                                                                                                                                  | Are they different?   |
|:-----------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------|
| `"how do transformers encode meaning?"`                          | Cross-encoders process the query and document simultaneously through a Transformer block, yielding highly accurate relevance scores but at a high computational cost. | Cross-encoders process the query and document simultaneously through a Transformer block, yielding highly accurate relevance scores but at a high computational cost. | No                    |
| `"optimization techniques for training"`                         | Learning rate schedulers, like Cosine Annealing, adjust the step size during training to help the optimizer escape local minima.                                      | AdamW is an optimization technique that decouples weight decay from the gradient update to improve model generalization.                                              | Yes                   |
| `"why are vector databases fast for nearest neighbor searches?"` | Vector databases index dense embeddings using algorithms like HNSW for fast approximate nearest neighbor search.                                                      | Vector databases index dense embeddings using algorithms like HNSW for fast approximate nearest neighbor search.                                                      | No                    |